In [79]:
import requests
import pandas as pd
import time
import ast
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [80]:
url = "https://api.opendota.com/api/publicMatches"
all_matches = []
params = {}
print("Pinging the OpenDota API...")

Pinging the OpenDota API...


In [ ]:
response = requests.get(url)
for i in range(100):
    response = requests.get(url, params=params)

    if response.status_code == 200:

        batch_data = response.json()
        all_matches.extend(batch_data)

        oldest_match_id = min([match["match_id"] for match in batch_data])

        params = {"less_than_match_id": oldest_match_id}

        print(f"Batch {i+1}/100 secured. Sleeping for 2 seconds...")

        time.sleep(2)

    else:
        print(
            f"Uh oh, the server got mad on batch {i+1}. Status code: {response.status_code}"
        )
        break

Batch 1/50 secured. Sleeping for 2 seconds...
Batch 2/50 secured. Sleeping for 2 seconds...
Batch 3/50 secured. Sleeping for 2 seconds...
Batch 4/50 secured. Sleeping for 2 seconds...
Batch 5/50 secured. Sleeping for 2 seconds...
Batch 6/50 secured. Sleeping for 2 seconds...
Batch 7/50 secured. Sleeping for 2 seconds...
Batch 8/50 secured. Sleeping for 2 seconds...
Batch 9/50 secured. Sleeping for 2 seconds...
Batch 10/50 secured. Sleeping for 2 seconds...
Batch 11/50 secured. Sleeping for 2 seconds...
Batch 12/50 secured. Sleeping for 2 seconds...
Batch 13/50 secured. Sleeping for 2 seconds...
Batch 14/50 secured. Sleeping for 2 seconds...
Batch 15/50 secured. Sleeping for 2 seconds...
Batch 16/50 secured. Sleeping for 2 seconds...
Batch 17/50 secured. Sleeping for 2 seconds...
Batch 18/50 secured. Sleeping for 2 seconds...
Batch 19/50 secured. Sleeping for 2 seconds...
Batch 20/50 secured. Sleeping for 2 seconds...
Batch 21/50 secured. Sleeping for 2 seconds...
Batch 22/50 secured. S

In [ ]:
df = pd.DataFrame(all_matches)

df = df.drop_duplicates(subset="match_id")

print(f"\nHarvest complete! We collected {len(df)} unique matches.")

df.to_csv("dota_matches.csv", index=False)
print("Data safely locked away in dota_5k_matches.csv")


Harvest complete! We collected 5000 unique matches.
Data safely locked away in dota_5k_matches.csv


In [ ]:
hero_data = requests.get("https://api.opendota.com/api/heroes").json()
# Create a dictionary mapping the integer ID directly to the English name
hero_dict = {hero["id"]: hero["localized_name"] for hero in hero_data}

In [ ]:
print("Loading the massive dataset...")
df_match = pd.read_csv("dota_matches.csv")

Loading the massive dataset...


In [ ]:
df_match["radiant_team"] = df_match["radiant_team"].apply(ast.literal_eval)
df_match["dire_team"] = df_match["dire_team"].apply(ast.literal_eval)

print(
    "Data loaded successfully! Here is what the target variable and features look like:"
)
display(df_match[["match_id", "radiant_win", "radiant_team", "dire_team"]].head())

Data loaded successfully! Here is what the target variable and features look like:


,match_id,radiant_win,radiant_team,dire_team
0,8715115098,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
1,8715114826,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
2,8715113819,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
3,8715113756,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"
4,8715113294,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]"


In [ ]:
print("Building the hero matrix... this might take a few seconds.")

matrix_rows = []

for i, row in df_match.iterrows():

    # Get rid of any data with 0 for the hero_id
    if 0 in row["radiant_team"] or 0 in row["dire_team"]:
        continue

    # Create dictionary for match
    match_features = {"match_id": row["match_id"], "radiant_win": row["radiant_win"]}

    # +1 for Radiant hero
    for hero_id in row["radiant_team"]:
        hero_name = hero_dict.get(hero_id, f"Unknown_Hero_{hero_id}")
        match_features[hero_name] = 1

    # -1 for Dire hero
    for hero_id in row["dire_team"]:
        hero_name = hero_dict.get(hero_id, f"Unknown_Hero_{hero_id}")
        match_features[hero_name] = -1

    matrix_rows.append(match_features)

final_df = pd.DataFrame(matrix_rows).fillna(0)

print("Matrix complete! Here is the shape of our new dataset (Rows, Columns):")
print(final_df.shape)
display(final_df.head())

Building the hero matrix... this might take a few seconds.
Matrix complete! Here is the shape of our new dataset (Rows, Columns):
(4982, 129)


,match_id,radiant_win,Rubick,Ogre Magi,Sniper,Lion,Spirit Breaker,Dazzle,Underlord,Techies,...,Winter Wyvern,Dark Seer,Earth Spirit,Dawnbreaker,Muerta,Bane,Ringmaster,Doom,Batrider,Pangolier
0,8715113008,True,1.0,1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,8715110205,False,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,8715109941,False,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,8715109493,False,1.0,0.0,1.0,0.0,0.0,-1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8715108678,False,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
X = final_df.drop(columns=["match_id", "radiant_win"])
y = final_df["radiant_win"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=12
)

print("Data successfully split!")
print(f"Training matches: {X_train.shape[0]}")
print(f"Testing matches: {X_test.shape[0]}")

Data successfully split!
Training matches: 3985
Testing matches: 997


In [ ]:
model = LogisticRegression()

model.fit(X_train, y_train)

preds = model.predict(X_test)

score = accuracy_score(y_test, preds)
print(f"Model Accuracy: {score * 100:.2f}%")

Model Accuracy: 56.97%


In [ ]:
coefficients = pd.DataFrame({"Hero": X_train.columns, "Weight": model.coef_[0]})

print("Top 5 Heroes the model thinks are strong(Highest Positive Weights):")
display(coefficients.sort_values(by="Weight", ascending=False).head(5))

print("\nTop 5 Heroes the model thinks are weak (Lowest Negative Weights):")
display(coefficients.sort_values(by="Weight", ascending=True).head(5))

Top 5 Heroes the model thinks are strong(Highest Positive Weights):


,Hero,Weight
29,Elder Titan,0.414400
97,Primal Beast,0.334450
61,Razor,0.322900
117,Winter Wyvern,0.308975
89,Phantom Lancer,0.294845



Top 5 Heroes the model thinks are weak (Lowest Negative Weights):


,Hero,Weight
119,Earth Spirit,-0.671541
62,Lycan,-0.569259
95,Chen,-0.514948
86,Naga Siren,-0.508284
78,Tinker,-0.486163


In [ ]:
print("Planting the Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=12, n_jobs=-1)

print("Training the trees ...")
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_score = accuracy_score(y_test, rf_predictions)

print("\n--- FINAL RESULTS ---")
print(f"New Random Forest Accuracy: {rf_score * 100:.2f}%")

importances = pd.DataFrame(
    {"Hero": X_train.columns, "Importance": rf_model.feature_importances_}
)

print("\nTop 5 Most Decisive Heroes (The ones swinging the win probability the most):")
display(importances.sort_values(by="Importance", ascending=False).head(5))

Planting the Random Forest...
Training the trees ...

--- FINAL RESULTS ---
New Random Forest Accuracy: 53.66%

Top 5 Most Decisive Heroes (The ones swinging the win probability the most):


,Hero,Importance
44,Pudge,0.016271
3,Lion,0.014595
2,Sniper,0.014055
19,Invoker,0.013883
53,Windranger,0.013659
